# PARC2026 Colab ブートストラップ

検証済みレシピ（`docs/env_setup.md` 参照、出典: [taku_sid氏のnote記事](https://note.com/taku_sid/n/n49a0008b29a6)）に基づく。
**ランタイム > ランタイムのタイプを変更 > GPU** を先に設定してから実行すること。

毎回のColabセッション開始時にこのノートブックの先頭セルから順に実行する。
ローカル(Claude Code / Codex CLI)で編集した自分のコード(`src/parc2026`)は別セルでimportする。

## 0. GPU確認

In [2]:
!nvidia-smi
!nvcc --version

Wed Jul 29 01:30:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Google Driveをマウント（モデル・アセットのキャッシュ永続化用）

Colabはセッションが切れるとローカルディスクが揮発し、モデルの再ダウンロードが発生する。
`HF_HOME` をDrive配下に向けてキャッシュを永続化する。

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/PARC2026'
os.makedirs(f'{DRIVE_ROOT}/hf_cache', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)

os.environ['HF_HOME'] = f'{DRIVE_ROOT}/hf_cache'
os.environ['MUJOCO_GL'] = 'egl'  # GPU描画に必須(評価実行前に毎回必要)

Mounted at /content/drive


## 2. 自分のリポジトリをclone / pull（ローカルと連携）

`parc2026` はprivateリポジトリのため、Colabからcloneするには認証が必要。
ColabのSecrets（鍵アイコン）に `GH_TOKEN`（GitHubのPersonal Access Token, repo scope）を登録してから
下のセルを実行する。ローカルで編集→push→ここで `git pull` すれば常に最新コードで実行できる。

In [4]:
from google.colab import userdata
import os

try:
    GH_TOKEN = userdata.get('GH_TOKEN')
    GITHUB_REPO_URL = f'https://{GH_TOKEN}@github.com/norikioka/parc2026.git'
    REPO_DIR = '/content/parc2026'

    if not os.path.exists(REPO_DIR):
        print(f"Cloning repository into {REPO_DIR}...")
        !git clone {GITHUB_REPO_URL} {REPO_DIR}
    else:
        print(f"Repository already exists. Pulling latest changes...")
        !cd {REPO_DIR} && git pull
except userdata.SecretNotFoundError:
    print("❌ エラー: 'GH_TOKEN' がColabのシークレットで見つかりません。")
    print("左メニューの鍵アイコンから 'GH_TOKEN' を登録し、ノートブックへのアクセスを許可してください。")

Cloning repository into /content/parc2026...
Cloning into '/content/parc2026'...
remote: Enumerating objects: 55, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 55 (delta 20), reused 49 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (55/55), 7.25 MiB | 28.13 MiB/s, done.
Resolving deltas: 100% (20/20), done.


## 3. LeRobot本体をclone + pi/libero extrasでインストール

PyTorchはColabにプリインストール済みのCUDA対応版をそのまま使う（`--index-url` での入れ直しは基本不要。
バージョン不一致エラーが出た場合のみ `docs/env_setup.md` の手順2を参照して入れ直す）。

In [5]:
LEROBOT_DIR = '/content/lerobot'
if not os.path.exists(LEROBOT_DIR):
    !git clone https://github.com/huggingface/lerobot.git {LEROBOT_DIR}

%cd {LEROBOT_DIR}
!pip install -q -e ".[pi,libero]"

Cloning into '/content/lerobot'...
remote: Enumerating objects: 45800, done.
remote: Counting objects: 100% (739/739), done.
remote: Compressing objects: 100% (365/365), done.
remote: Total 45800 (delta 600), reused 374 (delta 374), pack-reused 45061 (from 4)
Receiving objects: 100% (45800/45800), 225.81 MiB | 26.92 MiB/s, done.
Resolving deltas: 100% (28450/28450), done.
Filtering content: 100% (50/50), 69.11 MiB | 21.48 MiB/s, done.
/content/lerobot
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.7/217.7 kB 7.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.9/192.9 kB 10.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.8/164.8 kB 16.7 MB/s eta 0:00:00
  Preparing

## 4. HuggingFace認証（PaliGemmaはgated repoのため必須）

事前に https://huggingface.co で PaliGemma の利用規約に同意し、トークンを発行しておくこと。
Colabの場合は左メニューの鍵アイコン(Secrets)に `HF_TOKEN` を登録し、下のセルで読み込む方法が安全。

In [6]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

## 5. 【重要】無印LIBEROの評価を先に完了させる

LIBERO-Plusをインストールすると素のLIBEROが**置き換わって同時運用できなくなる**。
無印LIBEROでの評価・提出物確認が終わるまでは、次のセル（LIBERO-Plus導入）を実行しないこと。

## 6. LIBERO-Plusへの切り替え（無印LIBEROでの検証が終わってから）

In [7]:
LIBERO_PLUS_DIR = '/content/LIBERO-plus'
if not os.path.exists(LIBERO_PLUS_DIR):
    !git clone https://github.com/sylvestf/LIBERO-plus.git {LIBERO_PLUS_DIR}

%cd {LIBERO_PLUS_DIR}
!pip install -q --no-deps -e .
!pip install -q robosuite bddl easydict mujoco wand scikit-image gym

!hf download Sylvest/LIBERO-plus assets.zip --repo-type dataset --local-dir /content/LIBERO-plus_assets
!unzip -q /content/LIBERO-plus_assets/assets.zip -d {LIBERO_PLUS_DIR}/libero/libero

Cloning into '/content/LIBERO-plus'...
remote: Enumerating objects: 10782, done.
remote: Counting objects: 100% (211/211), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 10782 (delta 159), reused 139 (delta 139), pack-reused 10571 (from 1)
Receiving objects: 100% (10782/10782), 18.17 MiB | 24.26 MiB/s, done.
Resolving deltas: 100% (10315/10315), done.
Updating files: 100% (20964/20964), done.
/content/LIBERO-plus
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.0/145.0 kB 4.4 MB/s eta 0:00:00

assets.zip: downloading bytes:   3% 177M/6.40G [00:01<00:36, 170MB/s, 15.0MB/s  ]
assets.zip: downloading bytes:   4% 286M/6.40G [00:02<00:38, 158MB/s, 23.9MB/s  ]
assets.zip: downloading bytes:  27% 1.72G/6.40G [00:08<00:29, 159MB/s,  111MB/s  ]
assets.zip: downloading bytes:  29% 1.84G/6.40G [00:13<02:07, 35.6MB/s, 87.3MB/s  ]
assets.zip: downloading bytes:  29% 1.85G/6.40G [00:13<02:25, 31.2MB/s, 84.6MB/s  ]
assets.zip: downloading 

In [ ]:
!unzip -oq /content/LIBERO-plus_assets/assets.zip -d /content/LIBERO-plus/libero/libero
!ls /content/LIBERO-plus/libero/libero

In [ ]:
!ls /content/LIBERO-plus/libero/libero
!du -sh /content/LIBERO-plus/libero/libero

In [ ]:
import os
os.chdir('/content/LIBERO-plus')

from libero.libero import benchmark
bm = benchmark.get_benchmark_dict()
print(bm)

## 7. 自分のコード(src/parc2026)をimport

In [ ]:
import sys
sys.path.insert(0, f'{REPO_DIR}/src')

import parc2026
print('parc2026 import OK')

## 8. 動作確認(ここから先はタスクごとのノートブック/スクリプトへ)

詰まった手順・エラーメッセージは `docs/env_setup.md` の「つまずきポイント一覧」に追記していくこと。